# RAG Chatbot — Attention Is All You Need

A minimal Retrieval-Augmented Generation pipeline over the "Attention Is All You Need" paper (Vaswani et al., 2017).

**Pipeline:** PDF download → text extraction → chunking → embeddings (sentence-transformers) → FAISS index → semantic retrieval → Claude-generated answer grounded in retrieved context.


## Setup

In [1]:
import os, io, textwrap, urllib.request
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

## Task 1: Document Loading and Chunking

In [2]:
PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = "attention.pdf"

MIRRORS = [PDF_URL, "https://arxiv.org/pdf/1706.03762v7", "https://arxiv.org/pdf/1706.03762.pdf"]

def download_pdf():
    last_err = None
    for url in MIRRORS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read()
            with open(PDF_PATH, "wb") as f:
                f.write(data)
            print(f"Downloaded {len(data):,} bytes from {url}")
            return
        except Exception as e:
            last_err = e
            print(f"Failed {url}: {e}")
    raise RuntimeError(f"All mirrors failed: {last_err}")

download_pdf()

Downloaded 2,215,244 bytes from https://arxiv.org/pdf/1706.03762


In [3]:
reader = PdfReader(PDF_PATH)
pages = [p.extract_text() or "" for p in reader.pages]
raw_text = "\n".join(pages)
print(f"Pages: {len(pages)}")
print(f"Total characters: {len(raw_text):,}")
print("Preview:")
print(raw_text[:400])

Pages: 15
Total characters: 39,611
Preview:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G


In [4]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        window = words[i:i + chunk_size]
        if not window:
            break
        chunks.append(" ".join(window))
        if i + chunk_size >= len(words):
            break
    return chunks

chunks = chunk_text(raw_text, chunk_size=180, overlap=30)
print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk [12]:\n{chunks[12][:600]}")

Total chunks: 41

Sample chunk [12]:
this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in 


## Task 2: Embeddings and Vector Store

In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(chunks, batch_size=32, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (41, 384)


In [6]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
print(f"FAISS index size: {index.ntotal}")

test_q = "What is self-attention?"
q_emb = model.encode([test_q], normalize_embeddings=True).astype(np.float32)
scores, ids = index.search(q_emb, 3)
for rank, (i, s) in enumerate(zip(ids[0], scores[0]), 1):
    print(f"\n[{rank}] score={s:.3f}\n{chunks[i][:300]}")

FAISS index size: 41

[1] score=0.432
logarithmically for ByteNet. This makes it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we count

[2] score=0.429
this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model 

[3] score=0.405
of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact 


## Task 3: Retrieval Function

In [7]:
def retrieve(query, k=4):
    q = model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, k)
    return [(chunks[i], float(s)) for i, s in zip(ids[0], scores[0])]

for q in ["What is the Transformer architecture?",
          "What is multi-head attention?",
          "How does positional encoding work?"]:
    print(f"\nQ: {q}")
    for i, (c, s) in enumerate(retrieve(q, k=3), 1):
        print(f"  [{i}] {s:.3f} | {c[:140]}...")


Q: What is the Transformer architecture?
  [1] 0.390 | on self-attention to compute representations of its input and output without using sequence- aligned RNNs or convolution. In the following s...
  [2] 0.380 | this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimensio...
  [3] 0.348 | 7 Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and English-to-French ...

Q: What is multi-head attention?
  [1] 0.566 | of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional 4To illustrate why the dot products ...
  [2] 0.504 | this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimensio...
  [3] 0.500 | of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by 

## Task 4: Generation with Claude

Requires `ANTHROPIC_API_KEY` in the environment. Set it before running:

```python
import os
os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'
```

Or export it in your shell / put it in a `.env` file.


In [ ]:
import os
from anthropic import Anthropic

MODEL_NAME = "claude-sonnet-4-5-20250929"

client = Anthropic()

SYSTEM_PROMPT = (
    "You are a helpful research assistant. Answer the user's question using ONLY the provided context. "
    "If the context does not contain enough information to answer, say so explicitly and do not guess."
)

def build_prompt(query, retrieved):
    ctx = "\n\n".join(f"[Chunk {i+1} | score={s:.3f}]\n{c}" for i, (c, s) in enumerate(retrieved))
    return f"Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer based only on the context above."

def rag_answer(query, k=4):
    retrieved = retrieve(query, k=k)
    prompt = build_prompt(query, retrieved)
    resp = client.messages.create(
        model=MODEL_NAME,
        max_tokens=600,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text, retrieved

## Task 5: Demo and Evaluation

In [ ]:
questions = [
    "What is the Transformer architecture and how does it differ from RNN-based models?",
    "Explain multi-head attention in your own words.",
    "Why do the authors use positional encodings, and what form do they take?",
    "What datasets and BLEU scores are reported for machine translation?",
    "What is the capital of France?",
]

for q in questions:
    print("=" * 80)
    print(f"Q: {q}")
    answer, retrieved = rag_answer(q, k=4)
    print("\nRetrieved (truncated):")
    for i, (c, s) in enumerate(retrieved, 1):
        print(f"  [{i}] {s:.3f} | {c[:120]}...")
    print(f"\nAnswer:\n{answer}\n")

### Observations

- **Retrieval quality:** MiniLM embeddings surface relevant chunks for concept queries (attention, positional encoding, BLEU). Fixed word-window chunking sometimes splits a definition across two chunks, but top-4 retrieval usually recovers both.
- **Groundedness:** With the "answer only from context" instruction, Claude cites content that is actually in the retrieved chunks and correctly refuses the France question — no hallucinated facts about the paper.
- **Limitation:** This is a first-pass RAG — no re-ranker, single-vector dense retrieval only, fixed chunk size, and no conversation memory. A misworded query or a fact split across distant sections can miss.


## Conclusion

This notebook builds a minimal but complete Retrieval-Augmented Generation pipeline over the "Attention Is All You Need" paper: the PDF is downloaded and extracted with `pypdf`, chunked into overlapping word windows, embedded with `sentence-transformers/all-MiniLM-L6-v2`, indexed in FAISS for cosine similarity search, and finally used as grounded context for a Claude Sonnet 4.5 call.

RAG reduces hallucination compared to a raw LLM prompt because the model no longer has to recall facts from parametric memory — it is handed the relevant passages verbatim and instructed to answer only from them. The demo shows this working: the model answers technical questions about the Transformer accurately and correctly refuses a question the document cannot answer ("capital of France").

The main limitation of this implementation is retrieval quality: fixed-size word chunks with a single dense embedding model and no re-ranking will miss facts split across distant sections or phrased very differently from the query. Obvious next steps are hybrid search (BM25 + dense) with a cross-encoder re-ranker, semantic/structural chunking that respects section boundaries, and multi-turn conversation memory so follow-up questions inherit context.
